### 1. Setup

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import os
import warnings

warnings.filterwarnings('ignore')


### 2. Load Data

In [2]:
df = pd.read_csv('../data/processed/processed_data.csv')

print(f"Loaded {len(df):,} records")

df_sample = df.sample(n=min(5000, len(df)), random_state=42)
print(f"Sampled {len(df_sample):,} records")

Loaded 100,139 records
Sampled 5,000 records


### 3. Feature Engineering

In [4]:
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[18, 30, 40, 50, 60, 100],
    labels=['18-30', '31-40', '41-50', '51-60', '60+'],
    include_lowest=True
)

### 4. 3D Scatter

In [5]:
fig1 = px.scatter_3d(
    df_sample,
    x='Age',
    y='MonthlyBudget_USD',
    z='CustomerSatisfaction_pct',
    color='SkinType',
    size='ProductEffectiveness_Score',
    hover_data=['Gender', 'SkinConcerns'],
    title='3D: Age vs Budget vs Satisfaction'
)

fig1.update_layout(height=600)
fig1.show()

### 5. 3D SURFACE

In [6]:
age_bins = pd.cut(df['Age'], bins=6)
budget_bins = pd.cut(df['MonthlyBudget_USD'], bins=6)

pivot = df.pivot_table(
    values='CustomerSatisfaction_pct',
    index=age_bins,
    columns=budget_bins,
    aggfunc='mean'
)

fig2 = go.Figure(
    data=[
        go.Surface(z=pivot.values, colorscale='Viridis')
    ]
)

fig2.update_layout(
    title="Satisfaction Surface",
    height=600
)

fig2.show()

### 6. Grouped 2D bar

In [7]:
skin_dist = pd.crosstab(df['AgeGroup'], df['SkinType'], normalize='index') * 100
skin_dist = skin_dist.reset_index().melt(id_vars='AgeGroup')

fig3 = px.bar(
    skin_dist,
    x='SkinType',
    y='value',
    color='AgeGroup',
    barmode='group',
    title="Skin Type Distribution by Age Group"
)

fig3.show()

### 7. Heatmap

In [8]:
ingredients = [
    'UsesRetinol', 'UsesVitaminC', 'UsesNiacinamide',
    'UsesHyaluronicAcid', 'UsesSalicylicAcid', 'UsesCeramides'
]

ingredient_names = [
    'Retinol', 'Vitamin C', 'Niacinamide',
    'Hyaluronic Acid', 'Salicylic Acid', 'Ceramides'
]

available = [(i, n) for i, n in zip(ingredients, ingredient_names) if i in df.columns]

heatmap_data = []
for age in df['AgeGroup'].cat.categories:
    subset = df[df['AgeGroup'] == age]
    heatmap_data.append([subset[i].mean()*100 for i, _ in available])

fig4 = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=[n for _, n in available],
    y=list(df['AgeGroup'].cat.categories),
    colorscale='Viridis'
))

fig4.show()

### 8. Network Graph

In [9]:
available_cols = [i for i, _ in available]

matrix = df[available_cols].fillna(0).values
cooccur = np.dot(matrix.T, matrix)

G = nx.Graph()

for i, (_, name_i) in enumerate(available):
    G.add_node(name_i)

for i in range(len(available)):
    for j in range(i+1, len(available)):
        if cooccur[i, j] > cooccur.mean():
            G.add_edge(available[i][1], available[j][1])

pos = nx.spring_layout(G, seed=42)

edge_x, edge_y = [], []
for e in G.edges():
    x0, y0 = pos[e[0]]
    x1, y1 = pos[e[1]]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines')

node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers+text',
    text=list(G.nodes()),
    textposition='top center'
)

fig5 = go.Figure(data=[edge_trace, node_trace])
fig5.show()

###  9. Export

In [10]:
os.makedirs("../static", exist_ok=True)

for i, fig in enumerate([fig1, fig2, fig3, fig4, fig5, fig6 if 'fig6' in globals() else None]):
    if fig is not None:
        fig.write_html(f"../static/plot_{i+1}.html")

print("Export completed safely.")

Export completed safely.
